In [8]:
import json, re, glob, os
import pandas as pd

# --- edit this ---
RESULTS_DIR = "."          # folder containing the json files
FULL_LABEL  = "full"       # what to call the unnumbered run
# -----------------

pattern = re.compile(r"results_(nnunet|patchwork)(?:_(.+))?\.json$")

rows = []
for path in glob.glob(os.path.join(RESULTS_DIR, "results_*.json")):
    m = pattern.search(os.path.basename(path))
    if not m:
        continue
    model, size = m.group(1), m.group(2) or FULL_LABEL
    with open(path) as f:
        data = json.load(f)
    for cid, labels in data.items():
        for label, metrics in labels.items():
            rows.append({"case_id": cid, "label": label,
                         "model": model, "size": size, **metrics})

df = pd.concat([pd.DataFrame(rows)], ignore_index=True)

# size as ordered categorical: numeric sizes first (sorted), then alphabetical, then "full"
def _size_key(s):
    try:
        return (0, int(s), s)
    except ValueError:
        return (1, 0, s)

size_order = sorted([s for s in df["size"].unique() if s != FULL_LABEL], key=_size_key) + [FULL_LABEL]
df["size"] = pd.Categorical(df["size"], categories=size_order, ordered=True)

agg = (
    df.groupby(["model", "size", "label"], observed=True)
      .agg(dice_mean=("dice", "mean"), dice_std=("dice", "std"),
           nsd_mean=("nsd", "mean"),   nsd_std=("nsd", "std"),
           n=("dice", "count"))
      .reset_index()
)

print(f"{df['case_id'].nunique()} cases, {df['label'].nunique()} labels")
print("runs:", sorted({(m, str(s)) for m, s in zip(agg['model'], agg['size'])}))
agg.head()


290 cases, 25 labels
runs: [('nnunet', '20'), ('nnunet', '50'), ('nnunet', 'full'), ('nnunet', 'old'), ('patchwork', '50'), ('patchwork', 'full'), ('patchwork', 'original')]


,model,size,label,dice_mean,dice_std,nsd_mean,nsd_std,n
0,nnunet,20,AT_pelvis,0.958191,0.027517,NaN,NaN,290
1,nnunet,20,AT_thorax,0.931423,0.038740,NaN,NaN,290
2,nnunet,20,AT_upper_abdomen,0.934407,0.058599,NaN,NaN,290
3,nnunet,20,AVAT_pelvis,0.928786,0.048160,NaN,NaN,290
4,nnunet,20,AVAT_thorax,0.000000,0.000000,NaN,NaN,29


In [9]:
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

all_labels = sorted(agg["label"].unique())
all_models = sorted(agg["model"].unique())
all_sizes  = list(agg["size"].cat.categories)

# build all run combinations as (model, size) tuples
all_runs = [(m, s) for m in all_models for s in all_sizes
            if not agg[(agg["model"] == m) & (agg["size"] == s)].empty]
run_label = lambda m, s: f"{m} @ {s}"

# --- top controls ---
metric_dd = widgets.Dropdown(options=[("Dice", "dice"), ("NSD", "nsd")],
                             value="dice", description="Metric:")
sort_dd   = widgets.Dropdown(options=[("Score (desc)", "desc"),
                                      ("Score (asc)",  "asc"),
                                      ("Label name",   "name")],
                             value="desc", description="Sort by:")
err_cb    = widgets.Checkbox(value=True, description="Error bars (±std)")

# which runs to show
run_sel = widgets.SelectMultiple(
    options=[(run_label(m, s), (m, s)) for m, s in all_runs],
    value=tuple(all_runs),
    description="Runs:", rows=min(6, len(all_runs)),
    layout=widgets.Layout(width="260px"),
)

# --- label checkboxes ---
label_cbs = {lbl: widgets.Checkbox(value=True, description=lbl, indent=False,
                                   layout=widgets.Layout(width="auto"))
             for lbl in all_labels}
btn_all  = widgets.Button(description="All",  layout=widgets.Layout(width="70px"))
btn_none = widgets.Button(description="None", layout=widgets.Layout(width="70px"))

def set_all(value):
    for cb in label_cbs.values():
        cb.unobserve(render, names="value")
    for cb in label_cbs.values():
        cb.value = value
    for cb in label_cbs.values():
        cb.observe(render, names="value")
    render()

btn_all.on_click(lambda _: set_all(True))
btn_none.on_click(lambda _: set_all(False))

# --- mean summary widget ---
mean_html = widgets.HTML()

# --- figure: one trace per run, created up front so FigureWidget can update in place ---
fig = go.FigureWidget(data=[go.Bar(name=run_label(m, s), orientation="h")
                            for m, s in all_runs])
fig.update_layout(
    barmode="group",
    xaxis=dict(range=[0, 1.05]),
    yaxis=dict(autorange="reversed"),
    margin=dict(l=200, r=20, t=50, b=100),
    legend=dict(
        orientation="h",
        y=-0.15,
        yanchor="top",
        xanchor="center",
        x=0.5
    ),
)

def render(*_):
    metric = metric_dd.value
    mcol, scol = f"{metric}_mean", f"{metric}_std"

    selected_labels = [lbl for lbl, cb in label_cbs.items() if cb.value]
    selected_runs = set(run_sel.value)

    if not selected_labels or not selected_runs:
        with fig.batch_update():
            for trace in fig.data:
                trace.x, trace.y, trace.visible = [], [], False
            fig.layout.title = "Nothing to show"
        mean_html.value = ""
        return

    sub_agg = agg[agg["label"].isin(selected_labels)]

    # rank labels by mean across the selected runs only
    mask = sub_agg.apply(lambda r: (r["model"], r["size"]) in selected_runs, axis=1)
    rank = sub_agg[mask].groupby("label", observed=True)[mcol].mean()
    if   sort_dd.value == "desc": order = rank.sort_values(ascending=False).index
    elif sort_dd.value == "asc":  order = rank.sort_values(ascending=True).index
    else:                         order = sorted(rank.index)

    with fig.batch_update():
        for trace, (m, s) in zip(fig.data, all_runs):
            visible = (m, s) in selected_runs
            trace.visible = visible
            if not visible:
                continue
            sub = (sub_agg[(sub_agg["model"] == m) & (sub_agg["size"] == s)]
                   .set_index("label").reindex(order))
            trace.y = list(sub.index)
            trace.x = sub[mcol].tolist()
            trace.error_x = (dict(type="data", array=sub[scol].tolist())
                             if err_cb.value else dict(type="data", array=[]))
            trace.hovertemplate = ("<b>%{y}</b><br>" + metric +
                                   ": %{x:.3f}<extra>" + run_label(m, s) + "</extra>")
        fig.layout.title = (f"{metric.upper()} per label "
                            f"({len(order)}/{len(all_labels)} labels, "
                            f"{len(selected_runs)}/{len(all_runs)} runs)")
        fig.layout.height = max(350, 22 * len(order) + 100)

    # --- mean summary ---
    parts = []
    for m, s in all_runs:
        if (m, s) not in selected_runs:
            continue
        sub = sub_agg[(sub_agg["model"] == m) & (sub_agg["size"] == s)]
        mean_val = sub[mcol].mean()
        if not sub[mcol].isna().all():
            parts.append(f"<b>{run_label(m, s)}</b>: {mean_val:.3f}")
    label_note = f"{len(selected_labels)}/{len(all_labels)} labels"
    mean_html.value = (
        f"<div style='padding:4px 0; font-size:13px'>"
        f"Mean {metric.upper()} ({label_note}) &nbsp;|&nbsp; "
        + " &nbsp;&nbsp; ".join(parts)
        + "</div>"
    )

for w in (metric_dd, sort_dd, err_cb, run_sel):
    w.observe(render, names="value")
for cb in label_cbs.values():
    cb.observe(render, names="value")

checkbox_panel = widgets.VBox(
    list(label_cbs.values()),
    layout=widgets.Layout(max_height="500px", overflow_y="auto",
                          border="1px solid #ddd", padding="6px", width="280px"),
)
left_panel = widgets.VBox([
    widgets.HTML("<b>Runs</b>"),
    run_sel,
    widgets.HTML("<b>Labels</b>"),
    widgets.HBox([btn_all, btn_none]),
    checkbox_panel,
])

render()
display(widgets.VBox([
    widgets.HBox([metric_dd, sort_dd, err_cb]),
    mean_html,
    widgets.HBox([left_panel, fig]),
]))


In [7]:
import re
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

LC_DIR = Path("./learning_curves")

def _load_run(curve_dir: Path, model: str):
    vs = pd.read_csv(curve_dir / "valid_scalar.csv")
    if model == "nnunet":
        vs   = vs.set_index("epoch")
        dice = vs["mean_dice"]
    else:
        vs   = vs.set_index("step")
        vpc  = pd.read_csv(curve_dir / "valid_perclass.csv", index_col="step")
        class_cols = [c for c in vpc.columns
                      if c not in ("walltime_s", "walltime_h", "walltime_extrapolated")]
        dice = vpc[class_cols].mean(axis=1)
    return vs["walltime_h"], vs["loss"], dice

# discover & load all available curve directories
curves = {}
for d in sorted(LC_DIR.iterdir()):
    if not d.is_dir():
        continue
    m = re.match(r"(nnunet|patchwork)_(20|50|all)$", d.name)
    if not m:
        continue
    model, sz = m.group(1), "full" if m.group(2) == "all" else m.group(2)
    curves[(model, sz)] = _load_run(d, model)

all_lc_runs = sorted(curves.keys())
lc_run_label = lambda m, s: f"{m} @ {s}"

# widgets
lc_run_sel = widgets.SelectMultiple(
    options=[(lc_run_label(m, s), (m, s)) for m, s in all_lc_runs],
    value=tuple(all_lc_runs),
    description="Runs:", rows=len(all_lc_runs),
    layout=widgets.Layout(width="220px"),
)
lc_xaxis_dd = widgets.Dropdown(
    options=[("Wall time (h)", "walltime"), ("Epoch / step", "index")],
    value="walltime", description="X axis:",
)

# Two separate FigureWidgets — one trace per run, no explicit colors.
# Same fixed width/height and identical margins so the plot areas align.
# Legend is placed inside the dice plot (top-right) so it doesn't add height.
lc_fig_dice = go.FigureWidget()
lc_fig_loss = go.FigureWidget()

for model, sz in all_lc_runs:
    name = lc_run_label(model, sz)
    lc_fig_dice.add_trace(go.Scatter(name=name, mode="lines", showlegend=True))
    lc_fig_loss.add_trace(go.Scatter(name=name, mode="lines", showlegend=False))

_shared_layout = dict(
    autosize=False, width=500, height=380,
    margin=dict(l=60, r=20, t=40, b=50),
    xaxis=dict(title_text="Wall time (h)"),
)
lc_fig_dice.update_layout(**_shared_layout, title="Val Dice / F1",
                           yaxis_title="Mean Dice / F1",
                           legend=dict(x=1, y=1, xanchor="right", yanchor="top",
                                       bgcolor="rgba(255,255,255,0.7)"))
lc_fig_loss.update_layout(**_shared_layout, title="Val Loss",
                           yaxis_title="Loss", showlegend=False)

def _lc_render(*_):
    selected = set(lc_run_sel.value)
    use_time = lc_xaxis_dd.value == "walltime"
    xlabel   = "Wall time (h)" if use_time else "Epoch / step"
    with lc_fig_dice.batch_update(), lc_fig_loss.batch_update():
        for i, (model, sz) in enumerate(all_lc_runs):
            wt, loss, dice = curves[(model, sz)]
            x       = wt.tolist() if use_time else list(range(len(wt)))
            visible = (model, sz) in selected
            lc_fig_dice.data[i].x, lc_fig_dice.data[i].y = x, dice.tolist()
            lc_fig_loss.data[i].x, lc_fig_loss.data[i].y = x, loss.tolist()
            lc_fig_dice.data[i].visible = visible
            lc_fig_loss.data[i].visible = visible
        lc_fig_dice.layout.xaxis.title.text = xlabel
        lc_fig_loss.layout.xaxis.title.text = xlabel

lc_run_sel.observe(_lc_render,  names="value")
lc_xaxis_dd.observe(_lc_render, names="value")
_lc_render()

display(widgets.VBox([
    widgets.HBox([lc_run_sel, lc_xaxis_dd]),
    widgets.HBox([lc_fig_dice, lc_fig_loss]),
]))
